# Ex2_Joint-MHE
Jointly optimize states and parameters using the MHE objective; assess terminal PEM separately.

With `RUN_TRAINING = False`, load the pretrained parameters and display the model. View training history with `tensorboard --logdir runs`.


In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from data import load_data
from models import car_physics_raw_params, make_vehicle, parameter_dict
from estimation import (
    compute_mhe_estimations_multistart, make_mhe_config, tensor_parameters,
    terminal_excluded_ekf_warmstart,
)
from identification import PEMMHE, joint_mhe_components, one_step_pem
from utils import ROOT, load_initialization, save_joint_checkpoint, save_metrics, show_model


## 1. Configuration

In [ ]:
EXPERIMENT = 2
NAME = "Ex2_Joint-MHE"
RUN_TRAINING = False
SEED = 0
system = dict(nx=3, nu=2, nw=3, N=40, T=0.02,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="euler", rk4_steps=1, m=0.041, l1=0.029, l2=0.033)
REFERENCE_DIR = ROOT / "runs" / NAME
OUTPUT_DIR = ROOT / "runs" / "new" / (NAME + "_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
BATCH_SIZE, MAX_EPOCHS = 32, 2000
STATE_LR, PHYSICS_LR = 0.002, 1e-4
EVALUATION_INTERVAL, EARLY_STOPPING_EVALUATIONS = 10, 50

system["hidden_limit"] = 2.5


## 2. Data, model, and initial states

In [ ]:
data = load_data(EXPERIMENT)
initialization = load_initialization(NAME)
model = make_vehicle(system, initialization)
model.set_train_flag("physics")
model.sbar.requires_grad_(False)
model.Sigmainv_raw.requires_grad_(False)
if not RUN_TRAINING:
    checkpoint = torch.load(REFERENCE_DIR / "model_params_opt.pth", map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])

if RUN_TRAINING:
    initial_states = {s: terminal_excluded_ekf_warmstart(model, data[s])
                      for s in ("train", "val")}
    train_states = torch.nn.Parameter(torch.as_tensor(initial_states["train"]).clone())
    train_u, train_y = (torch.as_tensor(data["train"][key], dtype=torch.float32) for key in ("u", "y"))
    config = make_mhe_config(system, model, initialization, BATCH_SIZE)
    params = tensor_parameters(config)
    problem = PEMMHE(model, config, {"val": data["val"]})
    observed = data["train"]["y"].reshape(-1, 2)
    margin = .2*np.maximum(np.ptp(observed, axis=0), [.1 if EXPERIMENT==2 else 1., .2])
    vy_limit = max(2.5 if EXPERIMENT==2 else 3., 8*np.std(initial_states["train"][..., 1]))
    state_min = torch.tensor([observed[:,0].min()-margin[0], -vy_limit, observed[:,1].min()-margin[1]])
    state_max = torch.tensor([observed[:,0].max()+margin[0], vy_limit, observed[:,1].max()+margin[1]])


## 3. Joint objective

In [ ]:
if RUN_TRAINING:
    def objective(states):
        return joint_mhe_components(model, states, train_u, train_y,
            model.sbar, model.Sigmainv, params["mhe_Qinv_torch"], params["mhe_Rinv_torch"],
            params["C_torch"], sample_time=system["T"])

    if EXPERIMENT == 2:
        candidates = np.stack([terminal_excluded_ekf_warmstart(
            model, data["val"], initial_hidden_state=v) for v in (None, -2.5, 2.5)])
        compute_mhe_estimations_multistart(problem.observer, problem.loaders["val"], candidates)
    else:
        problem.estimate("val", initial_states["val"])


## 4. Optimize states and parameters

In [ ]:
if RUN_TRAINING:
    OUTPUT_DIR.mkdir(parents=True)
    physics = car_physics_raw_params(model)
    optimizer = torch.optim.Adam([{"params": [train_states], "lr": STATE_LR},
                                   {"params": physics, "lr": PHYSICS_LR}], amsgrad=True)
    best_val, stale = float("inf"), 0
    with SummaryWriter(OUTPUT_DIR) as writer:
        for epoch in range(MAX_EPOCHS + 1):
            if epoch:
                optimizer.zero_grad(set_to_none=True)
                objective(train_states)["total"].backward()
                torch.nn.utils.clip_grad_norm_(physics, 10.)
                torch.nn.utils.clip_grad_norm_([train_states], 100.)
                optimizer.step()
                with torch.no_grad():
                    train_states.clamp_(state_min, state_max)
            if epoch % EVALUATION_INTERVAL:
                continue
            problem.estimate("val")
            with torch.no_grad():
                train_pem = float(one_step_pem(model, train_states, train_u, train_y)[0])
                val_states = problem.loaders["val"].dataset.xhats
                val_u, val_y = (torch.as_tensor(data["val"][key], dtype=torch.float32) for key in ("u", "y"))
                val_pem = float(one_step_pem(model, val_states, val_u, val_y)[0])
            metrics = dict(train_pem=train_pem, val_pem=val_pem)
            if val_pem < best_val-1e-12:
                best_val, stale = val_pem, 0
                save_joint_checkpoint(
                    model,
                    train_states,
                    optimizer,
                    epoch,
                    metrics,
                    OUTPUT_DIR / "model_params_opt.pth"
                )
                save_metrics(OUTPUT_DIR, train_pem, val_pem, epoch)
                np.savez_compressed(OUTPUT_DIR / "mhe_states.npz", val=val_states.numpy())
            else:
                stale += int(epoch > 0)
            writer.add_scalar("PEM/train", train_pem, epoch)
            writer.add_scalar("PEM/val", val_pem, epoch)
            print(epoch, metrics)
            if stale >= EARLY_STOPPING_EVALUATIONS:
                break


## 5. Loaded model

In [ ]:
directory = OUTPUT_DIR if RUN_TRAINING else REFERENCE_DIR
checkpoint = torch.load(directory / "model_params_opt.pth", weights_only=False, map_location="cpu")
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"State dimension: {system['nx']}; input dimension: {system['nu']}; horizon: {system['N']}")
show_model(directory, model, parameters=parameter_dict(model))


State dimension: 3; input dimension: 2; horizon: 40
Checkpoint: runs/Ex2_Joint-MHE/model_params_opt.pth
CarDynamicsPytorch()
Model parameters: 23
Bf         5.9352522
Br         6.1424241
Cf         1.1940918
Cr         1.2160963
Df        0.20974751
Dr        0.18640545
Ef        0.01209259
Er      -0.027920008
Shf     0.0010112748
Shr    -0.0012634099
Svf   -0.00033700466
Svr    0.00032300502
Cm1       0.27533814
Cm2       0.04729566
Cr0      0.050024934
Cr2     0.0017284377
Iz     3.1984608e-05
